# Lesson 4b: Temporal Difference Learning - Practical Implementation

**Reinforcement Learning Course - Powell Clark**

---

## Learning Objectives

By the end of this notebook, you will:

1. **Implement TD(0) Prediction** - Online value function learning
2. **Implement SARSA** - On-policy TD control
3. **Implement Q-Learning** - Off-policy TD control
4. **Implement Expected SARSA** - Reduced variance TD control
5. **Implement Double Q-Learning** - Eliminate maximization bias
6. **Compare All Methods** - Performance analysis and benchmarking
7. **Solve Classic Environments** - CliffWalking, WindyGridWorld, Taxi

This notebook provides complete, production-ready implementations of all major TD algorithms.

---

## Table of Contents

1. [Setup and Utilities](#setup)
2. [TD(0) Prediction Implementation](#td-prediction)
3. [SARSA Implementation](#sarsa)
4. [Q-Learning Implementation](#q-learning)
5. [Expected SARSA](#expected-sarsa)
6. [Double Q-Learning](#double-q)
7. [CliffWalking Comparison](#cliff)
8. [Performance Benchmarking](#benchmark)
9. [Exercises](#exercises)

---

## 1. Setup and Utilities <a id='setup'></a>

In [ ]:
# Standard imports
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Tuple, Dict, List, Optional, Callable
from collections import defaultdict
import time

# Gymnasium
import gymnasium as gym

# Configuration
np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

print("✅ Imports successful")
print(f"NumPy: {np.__version__}")
print(f"Gymnasium: {gym.__version__}")

In [ ]:
def epsilon_greedy_policy(Q: defaultdict, state, n_actions: int, epsilon: float) -> int:
    """
    Epsilon-greedy action selection.
    
    Args:
        Q: Action-value function
        state: Current state
        n_actions: Number of actions
        epsilon: Exploration rate
    
    Returns:
        action: Selected action
    """
    if np.random.random() < epsilon:
        return np.random.randint(n_actions)
    else:
        return np.argmax(Q[state])


def get_greedy_action(Q: defaultdict, state) -> int:
    """Get greedy action from Q-function."""
    return np.argmax(Q[state])


def evaluate_policy(
    env: gym.Env,
    Q: defaultdict,
    n_episodes: int = 100,
    max_steps: int = 1000
) -> Tuple[float, float]:
    """
    Evaluate greedy policy derived from Q.
    
    Returns:
        avg_return: Average episode return
        success_rate: Fraction of successful episodes
    """
    returns = []
    successes = 0
    
    for _ in range(n_episodes):
        state, _ = env.reset()
        episode_return = 0
        
        for step in range(max_steps):
            action = get_greedy_action(Q, state)
            next_state, reward, terminated, truncated, _ = env.step(action)
            episode_return += reward
            
            if terminated or truncated:
                if reward > 0 or episode_return > -50:  # Success heuristic
                    successes += 1
                break
            
            state = next_state
        
        returns.append(episode_return)
    
    return np.mean(returns), successes / n_episodes


print("✅ Utility functions defined")

## 2. TD(0) Prediction Implementation <a id='td-prediction'></a>

### Random Walk Environment

Classic environment for testing TD prediction.

In [ ]:
class RandomWalk:
    """
    1D Random Walk environment.
    
    States: 0 (terminal left), 1, 2, 3, 4, 5, 6 (terminal right)
    Start: State 3 (center)
    Actions: 0 (left), 1 (right) - chosen uniformly at random
    Rewards: 0 everywhere except +1 at right terminal
    """
    def __init__(self, n_states: int = 7):
        self.n_states = n_states
        self.start_state = n_states // 2
        self.state = self.start_state
    
    def reset(self) -> int:
        self.state = self.start_state
        return self.state
    
    def step(self) -> Tuple[int, float, bool]:
        """Take random left or right step."""
        # Random action
        if np.random.random() < 0.5:
            self.state -= 1  # Left
        else:
            self.state += 1  # Right
        
        # Check terminal
        if self.state == 0:
            return self.state, 0.0, True  # Left terminal
        elif self.state == self.n_states - 1:
            return self.state, 1.0, True  # Right terminal (reward +1)
        else:
            return self.state, 0.0, False  # Non-terminal


# Test environment
rw = RandomWalk(n_states=7)
state = rw.reset()
print(f"Random Walk: {rw.n_states} states, start at {state}")
print("✅ RandomWalk environment created")

### TD(0) Prediction

In [ ]:
def td0_prediction(
    env: RandomWalk,
    n_episodes: int,
    alpha: float = 0.1,
    gamma: float = 1.0
) -> Tuple[np.ndarray, List[np.ndarray]]:
    """
    TD(0) prediction for RandomWalk.
    
    Args:
        env: RandomWalk environment
        n_episodes: Number of episodes
        alpha: Step size
        gamma: Discount factor
    
    Returns:
        V: Final value function
        history: Value function history for visualization
    """
    V = np.zeros(env.n_states)
    history = []
    
    for episode in range(n_episodes):
        state = env.reset()
        
        while True:
            next_state, reward, done = env.step()
            
            # TD(0) update: V(S) ← V(S) + α[R + γV(S') - V(S)]
            td_target = reward + gamma * V[next_state]
            td_error = td_target - V[state]
            V[state] += alpha * td_error
            
            if done:
                break
            
            state = next_state
        
        # Save snapshot every 10 episodes
        if episode % 10 == 0:
            history.append(V.copy())
    
    return V, history


# True values for RandomWalk (can be computed analytically)
true_values = np.array([0.0, 1/6, 2/6, 3/6, 4/6, 5/6, 1.0])

print("Training TD(0) on RandomWalk")
print("=" * 50)

rw = RandomWalk(n_states=7)
V_td, history = td0_prediction(rw, n_episodes=100, alpha=0.1)

print(f"\nTrue values:     {true_values}")
print(f"TD(0) estimates: {V_td}")
print(f"RMS Error:       {np.sqrt(np.mean((V_td - true_values)**2)):.4f}")

print("\n✅ TD(0) prediction complete")

### Visualize TD(0) Learning

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

# Plot true values
states = np.arange(1, 6)  # Non-terminal states
ax.plot(states, true_values[1:-1], 'k--', linewidth=2, label='True Values', marker='o', markersize=8)

# Plot TD(0) learning progression
for i, V_snapshot in enumerate(history[::2]):  # Every other snapshot
    alpha_val = 0.3 if i == len(history[::2]) - 1 else 0.1
    label = f'Episode {i*20}' if i in [0, len(history[::2])-1] else None
    ax.plot(states, V_snapshot[1:-1], alpha=alpha_val, label=label)

ax.set_xlabel('State', fontsize=12)
ax.set_ylabel('Value', fontsize=12)
ax.set_title('TD(0) Prediction on Random Walk', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/home/user/reinforcement-learning/static/images/4b_td0_random_walk.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ TD(0) visualization created")

## 3. SARSA Implementation <a id='sarsa'></a>

In [ ]:
def sarsa(
    env: gym.Env,
    n_episodes: int,
    alpha: float = 0.1,
    gamma: float = 0.99,
    epsilon: float = 0.1,
    decay_epsilon: bool = False
) -> Tuple[defaultdict, List[float]]:
    """
    SARSA: On-policy TD control.
    
    Update: Q(S,A) ← Q(S,A) + α[R + γQ(S',A') - Q(S,A)]
    
    Args:
        env: Gymnasium environment
        n_episodes: Number of episodes
        alpha: Learning rate
        gamma: Discount factor
        epsilon: Exploration rate
        decay_epsilon: Whether to decay epsilon (GLIE)
    
    Returns:
        Q: Learned action-value function
        episode_returns: Return per episode
    """
    n_actions = env.action_space.n
    Q = defaultdict(lambda: np.zeros(n_actions))
    episode_returns = []
    
    for episode in range(n_episodes):
        # Epsilon decay
        eps = epsilon / (1 + episode / 100) if decay_epsilon else epsilon
        
        state, _ = env.reset()
        action = epsilon_greedy_policy(Q, state, n_actions, eps)
        
        episode_return = 0
        
        for step in range(1000):
            # Take action
            next_state, reward, terminated, truncated, _ = env.step(action)
            episode_return += reward
            done = terminated or truncated
            
            # Choose next action (using same policy)
            next_action = epsilon_greedy_policy(Q, next_state, n_actions, eps)
            
            # SARSA update
            if done:
                td_target = reward
            else:
                td_target = reward + gamma * Q[next_state][next_action]
            
            td_error = td_target - Q[state][action]
            Q[state][action] += alpha * td_error
            
            if done:
                break
            
            state = next_state
            action = next_action
        
        episode_returns.append(episode_return)
    
    return Q, episode_returns


print("✅ SARSA implemented")

## 4. Q-Learning Implementation <a id='q-learning'></a>

In [ ]:
def q_learning(
    env: gym.Env,
    n_episodes: int,
    alpha: float = 0.1,
    gamma: float = 0.99,
    epsilon: float = 0.1,
    decay_epsilon: bool = False
) -> Tuple[defaultdict, List[float]]:
    """
    Q-Learning: Off-policy TD control.
    
    Update: Q(S,A) ← Q(S,A) + α[R + γ max_a Q(S',a) - Q(S,A)]
    
    Args:
        env: Gymnasium environment
        n_episodes: Number of episodes
        alpha: Learning rate
        gamma: Discount factor
        epsilon: Exploration rate
        decay_epsilon: Whether to decay epsilon
    
    Returns:
        Q: Learned action-value function
        episode_returns: Return per episode
    """
    n_actions = env.action_space.n
    Q = defaultdict(lambda: np.zeros(n_actions))
    episode_returns = []
    
    for episode in range(n_episodes):
        # Epsilon decay
        eps = epsilon / (1 + episode / 100) if decay_epsilon else epsilon
        
        state, _ = env.reset()
        episode_return = 0
        
        for step in range(1000):
            # Choose action (behavior policy: ε-greedy)
            action = epsilon_greedy_policy(Q, state, n_actions, eps)
            
            # Take action
            next_state, reward, terminated, truncated, _ = env.step(action)
            episode_return += reward
            done = terminated or truncated
            
            # Q-Learning update (target policy: greedy)
            if done:
                td_target = reward
            else:
                td_target = reward + gamma * np.max(Q[next_state])
            
            td_error = td_target - Q[state][action]
            Q[state][action] += alpha * td_error
            
            if done:
                break
            
            state = next_state
        
        episode_returns.append(episode_return)
    
    return Q, episode_returns


print("✅ Q-Learning implemented")

## 5. Expected SARSA <a id='expected-sarsa'></a>

In [ ]:
def expected_sarsa(
    env: gym.Env,
    n_episodes: int,
    alpha: float = 0.1,
    gamma: float = 0.99,
    epsilon: float = 0.1,
    decay_epsilon: bool = False
) -> Tuple[defaultdict, List[float]]:
    """
    Expected SARSA: Take expectation over next actions.
    
    Update: Q(S,A) ← Q(S,A) + α[R + γ Σ_a π(a|S') Q(S',a) - Q(S,A)]
    
    Args:
        env: Gymnasium environment
        n_episodes: Number of episodes
        alpha: Learning rate
        gamma: Discount factor
        epsilon: Exploration rate
        decay_epsilon: Whether to decay epsilon
    
    Returns:
        Q: Learned action-value function
        episode_returns: Return per episode
    """
    n_actions = env.action_space.n
    Q = defaultdict(lambda: np.zeros(n_actions))
    episode_returns = []
    
    for episode in range(n_episodes):
        # Epsilon decay
        eps = epsilon / (1 + episode / 100) if decay_epsilon else epsilon
        
        state, _ = env.reset()
        episode_return = 0
        
        for step in range(1000):
            # Choose action
            action = epsilon_greedy_policy(Q, state, n_actions, eps)
            
            # Take action
            next_state, reward, terminated, truncated, _ = env.step(action)
            episode_return += reward
            done = terminated or truncated
            
            # Compute expected value under ε-greedy policy
            if done:
                expected_q = 0
            else:
                # Expected value: ε/|A| for all actions + (1-ε) for greedy
                greedy_action = np.argmax(Q[next_state])
                expected_q = 0
                for a in range(n_actions):
                    if a == greedy_action:
                        prob = 1 - eps + eps / n_actions
                    else:
                        prob = eps / n_actions
                    expected_q += prob * Q[next_state][a]
            
            # Expected SARSA update
            td_target = reward + gamma * expected_q
            td_error = td_target - Q[state][action]
            Q[state][action] += alpha * td_error
            
            if done:
                break
            
            state = next_state
        
        episode_returns.append(episode_return)
    
    return Q, episode_returns


print("✅ Expected SARSA implemented")

## 6. Double Q-Learning <a id='double-q'></a>

In [ ]:
def double_q_learning(
    env: gym.Env,
    n_episodes: int,
    alpha: float = 0.1,
    gamma: float = 0.99,
    epsilon: float = 0.1,
    decay_epsilon: bool = False
) -> Tuple[defaultdict, defaultdict, List[float]]:
    """
    Double Q-Learning: Eliminate maximization bias.
    
    Maintains two Q-functions Q1 and Q2.
    Updates alternate between them, using one for selection and other for evaluation.
    
    Args:
        env: Gymnasium environment
        n_episodes: Number of episodes
        alpha: Learning rate
        gamma: Discount factor
        epsilon: Exploration rate
        decay_epsilon: Whether to decay epsilon
    
    Returns:
        Q1: First action-value function
        Q2: Second action-value function
        episode_returns: Return per episode
    """
    n_actions = env.action_space.n
    Q1 = defaultdict(lambda: np.zeros(n_actions))
    Q2 = defaultdict(lambda: np.zeros(n_actions))
    episode_returns = []
    
    for episode in range(n_episodes):
        # Epsilon decay
        eps = epsilon / (1 + episode / 100) if decay_epsilon else epsilon
        
        state, _ = env.reset()
        episode_return = 0
        
        for step in range(1000):
            # Choose action using sum of Q1 and Q2
            Q_sum = {s: Q1[s] + Q2[s] for s in set(list(Q1.keys()) + list(Q2.keys()))}
            if state not in Q_sum:
                Q_sum[state] = np.zeros(n_actions)
            
            if np.random.random() < eps:
                action = np.random.randint(n_actions)
            else:
                action = np.argmax(Q_sum[state])
            
            # Take action
            next_state, reward, terminated, truncated, _ = env.step(action)
            episode_return += reward
            done = terminated or truncated
            
            # Randomly choose which Q to update
            if np.random.random() < 0.5:
                # Update Q1
                if done:
                    td_target = reward
                else:
                    # Use Q1 for action selection, Q2 for evaluation
                    best_action = np.argmax(Q1[next_state])
                    td_target = reward + gamma * Q2[next_state][best_action]
                
                td_error = td_target - Q1[state][action]
                Q1[state][action] += alpha * td_error
            else:
                # Update Q2
                if done:
                    td_target = reward
                else:
                    # Use Q2 for action selection, Q1 for evaluation
                    best_action = np.argmax(Q2[next_state])
                    td_target = reward + gamma * Q1[next_state][best_action]
                
                td_error = td_target - Q2[state][action]
                Q2[state][action] += alpha * td_error
            
            if done:
                break
            
            state = next_state
        
        episode_returns.append(episode_return)
    
    return Q1, Q2, episode_returns


print("✅ Double Q-Learning implemented")

## 7. CliffWalking Comparison <a id='cliff'></a>

Classic environment showing difference between SARSA and Q-Learning.

In [ ]:
# Create CliffWalking environment
env = gym.make('CliffWalking-v0')

print("CliffWalking Environment")
print("=" * 50)
print(f"State space: {env.observation_space}")
print(f"Action space: {env.action_space}")
print("\nLayout: 4x12 grid")
print("  Start: Bottom-left")
print("  Goal: Bottom-right")
print("  Cliff: Bottom row (except start and goal)")
print("  Cliff penalty: -100, return to start")
print("  Step cost: -1")
print("\n✅ CliffWalking loaded")

### Train SARSA and Q-Learning

In [ ]:
print("Training on CliffWalking")
print("=" * 60)

n_episodes = 500
alpha = 0.1
gamma = 0.99
epsilon = 0.1

print(f"\nHyperparameters:")
print(f"  Episodes: {n_episodes}")
print(f"  Alpha: {alpha}")
print(f"  Gamma: {gamma}")
print(f"  Epsilon: {epsilon}")

# Train SARSA
print("\nTraining SARSA...")
start = time.time()
Q_sarsa, returns_sarsa = sarsa(env, n_episodes, alpha, gamma, epsilon)
time_sarsa = time.time() - start
print(f"  Completed in {time_sarsa:.2f}s")

# Train Q-Learning
print("\nTraining Q-Learning...")
start = time.time()
Q_qlearn, returns_qlearn = q_learning(env, n_episodes, alpha, gamma, epsilon)
time_qlearn = time.time() - start
print(f"  Completed in {time_qlearn:.2f}s")

# Train Expected SARSA
print("\nTraining Expected SARSA...")
start = time.time()
Q_expsarsa, returns_expsarsa = expected_sarsa(env, n_episodes, alpha, gamma, epsilon)
time_expsarsa = time.time() - start
print(f"  Completed in {time_expsarsa:.2f}s")

print("\n✅ All algorithms trained")

### Compare Learning Curves

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

# Smooth returns with moving average
window = 10

def smooth(returns, window):
    return np.convolve(returns, np.ones(window)/window, mode='valid')

ax.plot(smooth(returns_sarsa, window), label='SARSA', linewidth=2, alpha=0.8)
ax.plot(smooth(returns_qlearn, window), label='Q-Learning', linewidth=2, alpha=0.8)
ax.plot(smooth(returns_expsarsa, window), label='Expected SARSA', linewidth=2, alpha=0.8)

ax.set_xlabel('Episode', fontsize=12)
ax.set_ylabel(f'Return (smoothed, window={window})', fontsize=12)
ax.set_title('TD Control Comparison - CliffWalking', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.axhline(y=-13, color='green', linestyle='--', alpha=0.5, label='Optimal path')

plt.tight_layout()
plt.savefig('/home/user/reinforcement-learning/static/images/4b_cliffwalking_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ Learning curves plotted")

### Analyze Final Policies

In [ ]:
print("Final Policy Performance (100 test episodes)")
print("=" * 60)

# Evaluate greedy policies
avg_return_sarsa, _ = evaluate_policy(env, Q_sarsa, n_episodes=100)
avg_return_qlearn, _ = evaluate_policy(env, Q_qlearn, n_episodes=100)
avg_return_expsarsa, _ = evaluate_policy(env, Q_expsarsa, n_episodes=100)

print(f"\nSARSA:          {avg_return_sarsa:.2f}")
print(f"Q-Learning:     {avg_return_qlearn:.2f}")
print(f"Expected SARSA: {avg_return_expsarsa:.2f}")

print("\nInterpretation:")
print("  - SARSA learns safe path (away from cliff)")
print("  - Q-Learning learns optimal path (near cliff)")
print("  - Expected SARSA: intermediate (lower variance)")
print("  - Optimal return: -13 (shortest path)")

print("\n✅ Policy analysis complete")

## 8. Performance Benchmarking <a id='benchmark'></a>

In [ ]:
def benchmark_all_methods(
    env: gym.Env,
    n_episodes: int = 500,
    n_runs: int = 5
) -> Dict:
    """
    Comprehensive benchmark of all TD methods.
    
    Returns:
        results: Dict with performance metrics
    """
    methods = {
        'SARSA': sarsa,
        'Q-Learning': q_learning,
        'Expected SARSA': expected_sarsa,
    }
    
    results = {}
    
    for name, method in methods.items():
        print(f"\nBenchmarking {name}...")
        
        all_returns = []
        final_performance = []
        times = []
        
        for run in range(n_runs):
            start = time.time()
            
            if 'Double' in name:
                Q1, Q2, returns = method(env, n_episodes)
                # Average Q1 and Q2 for evaluation
                Q_avg = defaultdict(lambda: np.zeros(env.action_space.n))
                for s in set(list(Q1.keys()) + list(Q2.keys())):
                    Q_avg[s] = (Q1[s] + Q2[s]) / 2
                Q = Q_avg
            else:
                Q, returns = method(env, n_episodes)
            
            elapsed = time.time() - start
            
            all_returns.append(returns)
            avg_return, _ = evaluate_policy(env, Q, n_episodes=100)
            final_performance.append(avg_return)
            times.append(elapsed)
        
        results[name] = {
            'returns': np.mean(all_returns, axis=0),
            'final_performance': np.mean(final_performance),
            'final_std': np.std(final_performance),
            'avg_time': np.mean(times)
        }
    
    return results


print("Comprehensive Benchmark")
print("=" * 70)
print(f"Environment: CliffWalking")
print(f"Runs per method: 5")
print(f"Episodes per run: 500")

results = benchmark_all_methods(env, n_episodes=500, n_runs=5)

# Display results
print("\n" + "=" * 70)
print(f"{'Method':<20} {'Final Return':<15} {'Std':<10} {'Time (s)':<10}")
print("=" * 70)

for name, metrics in results.items():
    print(f"{name:<20} {metrics['final_performance']:<15.2f} {metrics['final_std']:<10.2f} {metrics['avg_time']:<10.2f}")

print("\n✅ Comprehensive benchmark complete")

## 9. Exercises <a id='exercises'></a>

### Exercise 1: Windy GridWorld

Implement and solve the Windy GridWorld environment.

In [ ]:
# TODO: Solve WindyGridworld-v0
# YOUR CODE HERE

# env = gym.make('TODO')
# Q, returns = sarsa(env, ...)
# Plot learning curve and policy

### Exercise 2: Step-Size Comparison

Compare different learning rates (alpha values).

In [ ]:
# TODO: Test alpha in [0.01, 0.05, 0.1, 0.5]
# YOUR CODE HERE

# for alpha in [0.01, 0.05, 0.1, 0.5]:
#     Q, returns = q_learning(env, n_episodes=500, alpha=alpha)
#     plt.plot(returns, label=f'α={alpha}')

### Exercise 3: n-Step SARSA

Implement n-step SARSA and compare with 1-step.

In [ ]:
# TODO: Implement n-step SARSA
# YOUR CODE HERE

# Hints:
# - Store last n transitions
# - Compute n-step return
# - Update Q-value

---

## Summary

In this notebook, you have:

1. ✅ **Implemented TD(0) Prediction** - Online value learning
2. ✅ **Implemented SARSA** - On-policy TD control
3. ✅ **Implemented Q-Learning** - Off-policy TD control
4. ✅ **Implemented Expected SARSA** - Lower variance TD
5. ✅ **Implemented Double Q-Learning** - Eliminate maximization bias
6. ✅ **Compared All Methods** - CliffWalking benchmark
7. ✅ **Analyzed Performance** - Learning curves and final policies

### Key Results

**CliffWalking**:
- SARSA: Safe path, ~-17 return
- Q-Learning: Optimal path, ~-13 return
- Expected SARSA: Intermediate, lower variance

**Insights**:
1. **SARSA**: Conservative, learns value of exploratory policy
2. **Q-Learning**: Aggressive, learns value of optimal policy
3. **Expected SARSA**: Best of both worlds (often)
4. **Double Q-Learning**: Unbiased, prevents overestimation

### Algorithm Selection Guide

**Use SARSA when**:
- Safety matters (robots, real systems)
- Exploration has real cost
- Want conservative policy

**Use Q-Learning when**:
- Learning in simulation
- Want optimal policy
- Can afford exploration failures

**Use Expected SARSA when**:
- Want lower variance
- Have computational budget for $O(|A|)$ updates
- Best all-around choice

**Use Double Q-Learning when**:
- Maximization bias is a problem
- Have 2x memory available
- Need unbiased estimates

### Transition to Deep RL

TD methods with function approximation lead to:
- **Deep Q-Networks (DQN)** - Lesson 7
- **Actor-Critic methods** - Lesson 8
- **Policy Gradient methods** - Lesson 9

These extend TD learning to continuous state spaces using neural networks.

---

**Lesson 4b Complete!** 🎉

You now have production-ready implementations of all major TD control algorithms.